In [37]:
import json
from pathlib import Path
import string

import duckdb
from openai import OpenAI
import pandas as pd
from pydantic import BaseModel

1. Iterate through responses to grab user annotations
2. For each newsletter, select top 2 stories
3. Run conditions - prompts, models, input text
4. LLM judge eval
5. Calculate error rate

In [2]:
con = duckdb.connect(database=':memory:')
client = OpenAI()

with open("../prompts/item_ranking.txt", "r") as f:
    prompt_rank = f.read()

with open("../prompts/framing.txt", "r") as f:
    prompt_write = f.read()

In [54]:
# response data
responses = con.execute(
    "SELECT user_annotations FROM '../data/responses_full.csv' WHERE user_annotations != 'not collected' AND user_annotations IS NOT NULL"
).fetchdf()

# newsletter data
data_path = Path("../data/emails")

newsletter_paths = [
    data_path
    / "Arizona and Missouri Will Vote on Abortion in November - The New York Times.csv",
    data_path / "Harris and Trump Battled for the Midwest - The New York Times.csv",
    data_path
    / "In Chicago, Obama Aims to Resurrect a Movement - The New York Times.csv",
    data_path
    / "Russia Freed Evan Gershkovich in a Major Prisoner Swap - The New York Times.csv",
    data_path / "Trump’s Pitch to Parents - The New York Times.csv",
]

# fulltext data
fulltext = pd.read_csv("../data/nyt_fulltext.csv", usecols=["url", "text"])

In [ ]:
# response models
class RankingResponse(BaseModel):
    rank1_id: str
    rank1_headline: str
    rank1_explanation: str
    rank2_id: str
    rank2_headline: str
    rank2_explanation: str

class HeadlineResponse(BaseModel):
    explanation: str
    headline: str
    subhed: str

class AccuracyEval(BaseModel):
    accuracy_rating: str
    reasoning: str
    issue_category: str

In [33]:
test_response = responses.user_annotations.iloc[0]
test_newsletter = newsletter_paths[0]
test_stories = con.execute(
    f"SELECT 'HEADLINE: ' || headline || '\n\nDESCRIPTION: ' || description AS newsletter_item, url FROM '{test_newsletter}' WHERE headline != 'SKIP' "
).fetchdf()

# Add an alphabetical letter ID for each item
test_stories["item_id"] = [string.ascii_uppercase[i] for i in range(len(test_stories))]
test_stories["newsletter_item"] = test_stories.apply(lambda row: f"{row['newsletter_item']}\n\nID: {row['item_id']}", axis=1)

test_day_of_week = pd.to_datetime(con.execute(f"SELECT publication_date FROM '{test_newsletter}' LIMIT 1").fetchdf().publication_date.iloc[0]).day_name()

In [40]:
format_input = f"""USER NOTES: {test_response}

CANDIDATE STORIES: {"\n".join(test_stories.newsletter_item.tolist())}"""
messages = [
    {"role": "system", "content": prompt_rank},
    {"role": "user", "content": format_input},
]
resp = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=messages,  # type: ignore
    response_format=RankingResponse,
    temperature=0.7,
)

test_rankings = resp.choices[0].message.content
test_rankings = json.loads(test_rankings)

In [41]:
test_rankings

{'rank1_id': 'A',
 'rank1_headline': 'Arizona and Missouri Will Vote on Abortion in November',
 'rank1_explanation': "This article aligns with the user's interest in U.S. politics and governance, specifically touching on social issues like abortion rights. The critical and reflective tone regarding state-level political decisions on a contentious issue matches the user's tone preference, and the political framing suits their interest in governance impacts on society.",
 'rank2_id': 'D',
 'rank2_headline': 'Summer heat can damage mailed medication',
 'rank2_explanation': "This story addresses social and economic issues related to healthcare, resonating with the user's interest in social issues and policies that impact daily life. The framing of a human interest narrative about the risks faced by individuals due to systemic challenges aligns with the user's preference for insightful, reflective content."}

In [46]:
test_items_for_rewrite = test_stories[test_stories.item_id.isin([test_rankings["rank1_id"], test_rankings["rank2_id"]])].newsletter_item.tolist()

In [55]:
format_input = f"""STORIES: {"\n".join(test_items_for_rewrite)}
DAY OF WEEK: {test_day_of_week}
"""

messages = [
    {"role": "system", "content": prompt_write},
    {"role": "user", "content": format_input},
]
resp = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=messages,  # type: ignore
    temperature=0.7,  # Add temperature
    response_format=HeadlineResponse
)

test_response = resp.choices[0].message.content

In [57]:
test_response = json.loads(test_response)

# Evaluation

In [49]:
with open("../prompts/evals/accuracy.md", "r") as f:
    prompt_accuracy = f.read()

In [58]:
def map_id_to_fulltext(item_id: str) -> str:
    """Map item ID to fulltext."""
    url = test_stories[test_stories.item_id == item_id].url.iloc[0]
    return fulltext[fulltext.url == url].text.iloc[0]

In [60]:
item1_fulltext = map_id_to_fulltext(test_rankings["rank1_id"])

In [63]:
test_eval = client.responses.parse(
    model="gpt-4.1-2025-04-14",
    instructions=prompt_accuracy,
    input=item1_fulltext,
    text_format=AccuracyEval
)

In [65]:
test_eval.output_parsed

AccuracyEval(accuracy_rating='Pass', reasoning="The headline accurately reflects the main focus of the story, which is Arizona's certification of signatures for an abortion rights constitutional amendment ballot measure and the potential impact on Democratic voter turnout.", issue_category='N/A')

In [66]:
item2_fulltext = map_id_to_fulltext(test_rankings["rank2_id"])

In [67]:
test_eval2 = client.responses.parse(
    model="gpt-4.1-2025-04-14",
    instructions=prompt_accuracy,
    input=item2_fulltext,
    text_format=AccuracyEval
)

In [68]:
test_eval2.output_parsed

AccuracyEval(accuracy_rating='Pass', reasoning='The headline accurately summarizes the main focus of the article, which is that rising summer temperatures are threatening the efficacy of mail-order medications due to a lack of appropriate temperature regulations and safeguards during shipping.', issue_category='N/A')